In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load the cleaned dataset
df = pd.read_csv('../data/interim/cleaned_telco_churn.csv')

print(f"Dataset shape: {df.shape}")
print("Churn distribution:")
print(df['Churn'].value_counts())

Dataset shape: (7032, 21)
Churn distribution:
Churn
No     5163
Yes    1869
Name: count, dtype: int64


### Feature engineering has well defined categories. There categories include:
| Category | Description |
|--------------| ----------------------|
| Decomposition | Breaking one column into many logical parts |
| Aggregation | Combining multiple columns to obtain one logical column |
| Interaction Features | Multiplying or combining two features to capture their joint effect |
| Ratio Features | One column divided by another |
| Encoding | Converting categoricals to numbers |
| Binning | Continuous to discrete groups |
| Flag / Indicator | Binning signal for a condition |
| Domain Features | Features built from business knowledge not just mathematical transformation |

In [4]:
# Tenure group - loyalty stage segmentation
df['tenure_group'] = pd.cut(
    df['tenure'],

    bins = [0, 12, 24, 48, 60, 72, 84],
    labels = [
        "0-12 months",
        "13-24 months",
        "25-48 months",
        "49-60 months", 
        "61-72 months",
        "73-84 months"
    ], include_lowest=True
)

print(df['tenure_group'].value_counts().sort_index())

tenure_group
0-12 months     2175
13-24 months    1024
25-48 months    1594
49-60 months     832
61-72 months    1407
73-84 months       0
Name: count, dtype: int64


In [5]:
# average monthly spend - more stable than MonthlyCharges
# avoid division by zero for tenure = 0 
df["avg_monthly_spend"] = np.where(
    df['tenure'] > 0,
    df['TotalCharges'] / df['tenure'],
    df['MonthlyCharges']
)

print(df['avg_monthly_spend'].describe())

count    7032.000000
mean       64.799424
std        30.185891
min        13.775000
25%        36.179891
50%        70.373239
75%        90.179560
max       121.400000
Name: avg_monthly_spend, dtype: float64


In [9]:
# Has no support safety net - likely to churn if they have an issue
df['has_no_support'] = (
    (df['TechSupport'] == 'No') &
    (df['OnlineSecurity'] == 'No') 
).astype(int)

print(df['has_no_support'].value_counts())
print(f'\nChurn rates by support status:')
print(df.groupby('has_no_support')['Churn'].value_counts(normalize=True).unstack())

has_no_support
0    4479
1    2553
Name: count, dtype: int64

Churn rates by support status:
Churn                No      Yes
has_no_support                  
0               0.86180  0.13820
1               0.51038  0.48962


In [14]:
# count of additional services - more services may indicate higher engagement and lower churn risk
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

df['num_services'] = df[service_cols] \
        .apply(lambda row: sum(val == 'Yes' for val in row), axis=1)

print(df['num_services'].value_counts().sort_index())
print(f'\nChurn rates by number of services:')
print(df.groupby('num_services')['Churn'].value_counts(normalize=True).unstack())

num_services
0    2213
1     966
2    1033
3    1117
4     850
5     569
6     284
Name: count, dtype: int64

Churn rates by number of services:
Churn               No       Yes
num_services                    
0             0.785359  0.214641
1             0.542443  0.457557
2             0.641820  0.358180
3             0.726052  0.273948
4             0.776471  0.223529
5             0.875220  0.124780
6             0.947183  0.052817


In [10]:
# high value customers - those with high total spend may be less likely to churn
spend_75th = df['MonthlyCharges'].quantile(0.75)
df['is_high_value'] = (df['MonthlyCharges'] > spend_75th).astype(int)

print(f"75th percentile of MonthlyCharges: {spend_75th:.2f}")
print(f"\nHigh value customers: {df['is_high_value'].sum()}")
print(f'\nChurn rates by high value status:')
print(df.groupby('is_high_value')['Churn'].value_counts(normalize=True).unstack())

75th percentile of MonthlyCharges: 89.86

High value customers: 1758

Churn rates by high value status:
Churn                No       Yes
is_high_value                    
0              0.755214  0.244786
1              0.671217  0.328783


In [12]:
# contract type - month-to-month customers are often more likely to churn than those with longer-term contracts
contract_risk_map = {
    'Two year': 0,
    'One year': 1,  
    'Month-to-month': 2
}

df['contract_risk'] = df['Contract'].map(contract_risk_map)

print(df['contract_risk'].value_counts().sort_index())
print(f'\nChurn rates by contract risk level:')
print(df.groupby('contract_risk')['Churn'].value_counts(normalize=True).unstack())

contract_risk
0    1685
1    1472
2    3875
Name: count, dtype: int64

Churn rates by contract risk level:
Churn                No       Yes
contract_risk                    
0              0.971513  0.028487
1              0.887228  0.112772
2              0.572903  0.427097
